# Simulated Annealing 

Simulated Annealing (SA) is a metaheuristic algorithm for global optimization. 
It is inspired by the physical process of annealing in metallurgy, where a material is heated and then slowly cooled to decrease defects and find a low-energy state.

### How it works:
Unlike simple Hill Climbing, SA can accept **worse solutions** with a certain probability. 
This probability depends on two main factors:
1. The difference in quality between the new state and the current state ($\Delta E$)
2. The current temperature ($T$)

The acceptance probability is calculated using the Boltzmann distribution:

$$
P(\text{accept}) = e^{-\frac{\Delta E}{T}}
$$

As the temperature $T$ cools down over time (according to a cooling schedule), the algorithm becomes less likely to accept worse moves, eventually behaving like standard Hill Climbing.


### Step 1: Define a Multimodal Objective Function

To show the advantage of Simulated Annealing over Hill Climbing, we need a function with **multiple local optima** (hills and valleys).
We will use a function with a global maximum and several local maxima:

$$
f(x) = x \cdot \sin(x)
$$

Our goal is to **maximize** this function within a specific search range, say $[0, 10]$.


In [1]:
import math
import random

def objective_function(x):
    # To prevent math domain errors, we keep x within bounds later
    return x * math.sin(x)


### Step 2: Implement the Simulated Annealing Algorithm

We write a function that takes:
- `objective_fn`: The function to maximize
- `bounds`: A tuple representing $[min\_val, max\_val]$
- `init_temp`: Starting temperature (high value)
- `cooling_rate`: Factor by which temperature is multiplied (e.g., $0.95$)
- `step_size`: How far we can jump to find a neighbor


In [5]:
def simulated_annealing(
    objective_fn, bounds, init_temp=150.0, cooling_rate=0.985, step_size=1.5
):
    # Start at a random position
    current = random.uniform(bounds[0], bounds[1])
    current_value = objective_fn(current)

    best = current
    best_value = current_value

    temp = init_temp
    iteration = 0

    print(f"Starting SA at x = {current:.4f}, f(x) = {current_value:.4f}\n")

    # Cool down loop
    while temp > 0.005:
        # Candidate step: normal distribution centered at current
        neighbor = random.normalvariate(current, step_size)
        neighbor = max(bounds[0], min(bounds[1], neighbor))  # Keep within bounds

        neighbor_value = objective_fn(neighbor)
        delta_e = neighbor_value - current_value

        # Accept rule
        if delta_e > 0:
            current = neighbor
            current_value = neighbor_value
        else:
            # Acceptance probability calculation
            prob = math.exp(delta_e / temp)
            if random.random() < prob:
                current = neighbor
                current_value = neighbor_value

        # Track global best
        if current_value > best_value:
            best = current
            best_value = current_value

        # Cool temperature
        temp *= cooling_rate
        iteration += 1

        if iteration % 40 == 0:
            print(
                f"Iter {iteration:03d} | Temp: {temp:.2f} | Current x: {current:.4f} | Best f(x): {best_value:.4f}"
            )

    return best, best_value


### Step 3: Run the Simulated Annealing Algorithm
We define the bounds to be $[0, 10]$ and run the algorithm.


In [6]:
bounds = (0.0, 10.0)
random.seed(42)  # For reproducibility

best_x, best_val = simulated_annealing(
    objective_function,
    bounds,
    init_temp=150.0,
    cooling_rate=0.985,
    step_size=1.8,
)

print("-" * 45)
print(f"Global Best Solution Found: x = {best_x:.4f}")
print(f"Global Best Value: f(x) = {best_val:.4f}")


Starting SA at x = 6.3943, f(x) = 0.7088

Iter 040 | Temp: 81.95 | Current x: 2.9236 | Best f(x): 6.6622
Iter 080 | Temp: 44.77 | Current x: 5.4830 | Best f(x): 6.6622
Iter 120 | Temp: 24.46 | Current x: 0.6882 | Best f(x): 7.5986
Iter 160 | Temp: 13.36 | Current x: 3.5505 | Best f(x): 7.8449
Iter 200 | Temp: 7.30 | Current x: 0.7941 | Best f(x): 7.8449
Iter 240 | Temp: 3.99 | Current x: 7.6421 | Best f(x): 7.9107
Iter 280 | Temp: 2.18 | Current x: 7.6362 | Best f(x): 7.9107
Iter 320 | Temp: 1.19 | Current x: 7.1166 | Best f(x): 7.9107
Iter 360 | Temp: 0.65 | Current x: 7.7273 | Best f(x): 7.9107
Iter 400 | Temp: 0.36 | Current x: 7.5033 | Best f(x): 7.9107
Iter 440 | Temp: 0.19 | Current x: 8.0765 | Best f(x): 7.9107
Iter 480 | Temp: 0.11 | Current x: 7.9289 | Best f(x): 7.9107
Iter 520 | Temp: 0.06 | Current x: 8.0436 | Best f(x): 7.9107
Iter 560 | Temp: 0.03 | Current x: 8.1585 | Best f(x): 7.9107
Iter 600 | Temp: 0.02 | Current x: 7.8548 | Best f(x): 7.9107
Iter 640 | Temp: 0.01 | 

### Compare with Standard Hill Climbing

Let's see what happens if we run standard Hill Climbing on the same function starting from $x=1.0$. 
It will likely get stuck in the nearest local maximum.


In [4]:
def hill_climbing_local(objective_fn, start, bounds, step_size=0.1):
    current = start
    current_value = objective_fn(current)

    while True:
        # Check left and right neighbors
        left_neighbor = max(bounds[0], current - step_size)
        right_neighbor = min(bounds[1], current + step_size)

        left_val = objective_fn(left_neighbor)
        right_val = objective_fn(right_neighbor)

        # Select the best neighbor
        if left_val > right_val:
            best_n, best_n_val = left_neighbor, left_val
        else:
            best_n, best_n_val = right_neighbor, right_val

        # If no improvement, stop
        if best_n_val <= current_value:
            break

        current = best_n
        current_value = best_n_val

    return current, current_value


# Run Hill Climbing starting at x = 1.0
hc_x, hc_val = hill_climbing_local(objective_function, start=1.0, bounds=bounds)
print(f"Hill Climbing result (starting at 1.0): x = {hc_x:.4f}, f(x) = {hc_val:.4f}")


Hill Climbing result (starting at 1.0): x = 2.0000, f(x) = 1.8186



- **Hill Climbing** starting at $x=1.0$ gets trapped at the first peak around $x \approx 2.03$ (where $f(x) \approx 1.82$).
- **Simulated Annealing** uses high temperatures early on to jump over the valley and successfully reaches the global optimum near $x \approx 7.97$ (where $f(x) \approx 7.91$).
